In [1]:
#!unzip -q ~/code/Maelle05/DyslexIA/data/data.zip '*T4*' -d ~/code/Maelle05/DyslexIA/data/data_T4 && rm -rf ~/code/Maelle05/DyslexIA/data/data_T4/__MACOSX

In [2]:
import os
import glob
import pandas as pd
import numpy as np
pd.set_option('display.max_columns', None)

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import xgboost as xgb
from sklearn.metrics import accuracy_score, recall_score, precision_score
from sklearn.inspection import permutation_importance
from sklearn.model_selection import StratifiedKFold

import matplotlib.pyplot as plt

In [3]:
DATA_PATH = "/home/justine/code/Maelle05/DyslexIA/data/data_T4/data"
files_metrics = glob.glob(os.path.join(DATA_PATH, "*metrics.csv"))
files_fixations = glob.glob(os.path.join(DATA_PATH, "*fixations.csv"))

In [4]:
fix_dict = {}

for f in files_fixations:
    df_f = pd.read_csv(f)
    sid = int(df_f["sid"].iloc[0])

    fix_dict[sid] = {
        "std_fixation_duration": df_f["duration_ms"].std(),
        "mean_fixation_x": df_f["fix_x"].mean(),
        "mean_fixation_y": df_f["fix_y"].mean(),
        "dispersion_x" : df_f["fix_x"].std(),
        "dispersion_y" : df_f["fix_y"].std(),
        "n_lines_fixated": df_f["aoi_line"].nunique()
    }


rows = []

for f in files_metrics:
    df_m = pd.read_csv(f)
    sid = int(df_m["sid"].iloc[0])

    row = {"sid": sid}
    row.update(df_m[["mean_fix_dur_trial", "n_fix_trial", "mean_sacc_ampl_trial"]].iloc[0].to_dict())
    row.update(fix_dict.get(sid, {}))

    row["fixation_per_line_ratio"] = row.get("n_fix_trial") / row.get("n_lines_fixated")

    rows.append(row)

dataset = pd.DataFrame(rows)

dataset["n_fix_trial"] = dataset["n_fix_trial"].astype(int)
dataset = dataset.drop(columns=["n_lines_fixated"])

dataset.head()

,sid,mean_fix_dur_trial,n_fix_trial,mean_sacc_ampl_trial,std_fixation_duration,mean_fixation_x,mean_fixation_y,dispersion_x,dispersion_y,fixation_per_line_ratio
0,1040,309.317832,196,100.362344,184.015050,853.785638,476.846939,342.986401,174.168448,28.000000
1,1033,340.685731,167,96.923166,207.159492,828.402650,473.398204,355.983670,176.933267,23.857143
2,1300,541.112113,391,55.822048,498.007097,850.527986,473.088235,338.794864,172.425389,55.857143
3,1058,283.695312,157,138.152899,193.734388,843.326783,476.257962,353.848782,187.641413,22.428571
4,1476,568.633427,391,51.151222,568.816212,850.726029,409.794118,385.987378,168.457597,55.857143


Les features du papier OpenReview sont :

- Mean fixation duration
- Fixation count
- Mean saccade length
- Std fixation duration
- Mean fixation X
- Mean fixation Y
- Dispersion X
- Dispersion Y
- Fixation-per-line ratio

In [5]:
labels = pd.read_csv("/home/justine/code/Maelle05/DyslexIA/data/dyslexia_class_label.csv")

dataset = dataset.merge(labels[["subject_id", "class_id"]], left_on="sid", right_on="subject_id").drop(columns=["subject_id"])
dataset.head()

,sid,mean_fix_dur_trial,n_fix_trial,mean_sacc_ampl_trial,std_fixation_duration,mean_fixation_x,mean_fixation_y,dispersion_x,dispersion_y,fixation_per_line_ratio,class_id
0,1040,309.317832,196,100.362344,184.015050,853.785638,476.846939,342.986401,174.168448,28.000000,0
1,1033,340.685731,167,96.923166,207.159492,828.402650,473.398204,355.983670,176.933267,23.857143,0
2,1300,541.112113,391,55.822048,498.007097,850.527986,473.088235,338.794864,172.425389,55.857143,1
3,1058,283.695312,157,138.152899,193.734388,843.326783,476.257962,353.848782,187.641413,22.428571,0
4,1476,568.633427,391,51.151222,568.816212,850.726029,409.794118,385.987378,168.457597,55.857143,1


In [6]:
dataset["class_id"].value_counts()

class_id
0    35
1    35
Name: count, dtype: int64

In [7]:
#X = dataset.drop(columns=["sid", "class_id"]).values
#X = dataset.drop(columns=["sid", "std_fixation_duration", "mean_fixation_x", "dispersion_x", "dispersion_y", "fixation_per_line_ratio", "class_id"]).values
#X = dataset.drop(columns=["sid", "mean_sacc_ampl_trial", "mean_fixation_y", "std_fixation_duration", "mean_fixation_x", "dispersion_x", "dispersion_y", "fixation_per_line_ratio", "class_id"]).values
X = dataset.drop(columns=["sid", "mean_fix_dur_trial", "mean_sacc_ampl_trial", "mean_fixation_y", "std_fixation_duration", "mean_fixation_x", "dispersion_x", "dispersion_y", "fixation_per_line_ratio", "class_id"]).values

#features_names = dataset.drop(columns=["sid", "class_id"]).columns
#features_names = dataset.drop(columns=["sid", "std_fixation_duration", "mean_fixation_x", "dispersion_x", "dispersion_y", "fixation_per_line_ratio", "class_id"]).columns
#features_names = dataset.drop(columns=["sid", "mean_sacc_ampl_trial", "mean_fixation_y", "std_fixation_duration", "mean_fixation_x", "dispersion_x", "dispersion_y", "fixation_per_line_ratio", "class_id"]).columns
features_names = dataset.drop(columns=["sid", "mean_fix_dur_trial", "mean_sacc_ampl_trial", "mean_fixation_y", "std_fixation_duration", "mean_fixation_x", "dispersion_x", "dispersion_y", "fixation_per_line_ratio", "class_id"]).columns

y = dataset["class_id"].values

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

#Standardisation à garder pour que xgboost ?
scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

imbalance_ratio = (y_train == 0).sum() / (y_train == 1).sum() #ici imbalance_ratio = 1

model = xgb.XGBClassifier(
    learning_rate=0.05,
    max_depth=2,
    n_estimators=50,
    reg_lambda=1.0,
    reg_alpha=0.1,
    min_child_weight=3,
    scale_pos_weight=imbalance_ratio, # pénalise le modèle pour les faux négatifs sur la classe 1 // default = 1
    random_state=42,
)

model.fit(X_train, y_train)

y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

train_acc = accuracy_score(y_train, y_train_pred)
test_acc = accuracy_score(y_test, y_test_pred)
test_acc

0.7857142857142857

In [9]:
test_rec = recall_score(y_test, y_test_pred)
test_prec = precision_score(y_test, y_test_pred)
display(test_rec)
display(test_prec)

0.8571428571428571

0.75

In [16]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)): #attention X pas standardisé

    model_cv = xgb.XGBClassifier(
        learning_rate=0.05,
        max_depth=2,
        n_estimators=50,
        reg_lambda=1.0,
        reg_alpha=0.1,
        min_child_weight=3,
        scale_pos_weight=(y[tr_idx] == 0).sum() / (y[tr_idx] == 1).sum(),
        random_state=42
        )

    model_cv.fit(X[tr_idx], y[tr_idx])
    fold_acc = accuracy_score(y[val_idx], model_cv.predict(X[val_idx]))
    cv_scores.append(fold_acc)
    print(f"  Fold {fold+1}: {fold_acc:.4f}")

print(f"  Mean ± Std: {np.mean(cv_scores):.4f} ± {np.std(cv_scores):.4f}")

  Fold 1: 0.6429
  Fold 2: 0.9286
  Fold 3: 0.6429
  Fold 4: 0.8571
  Fold 5: 0.7143
  Mean ± Std: 0.7571 ± 0.1161


In [17]:
thresholds = [0.10, 0.20, 0.30, 0.40, 0.50]
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for thresh in thresholds:
    recalls, precisions = [], []
    for train_idx, val_idx in skf.split(X, y):

        m = xgb.XGBClassifier(
            learning_rate=0.05,
            max_depth=2,
            n_estimators=50,
            reg_lambda=1.0,
            reg_alpha=0.1,
            min_child_weight=3,
            scale_pos_weight=(y[tr_idx] == 0).sum() / (y[tr_idx] == 1).sum(),
            random_state=42
            )

        m.fit(X[train_idx], y[train_idx])

        p = m.predict_proba(X[val_idx])[:, 1]
        preds = (p >= thresh).astype(int)

        recalls.append(recall_score(y[val_idx], preds))
        precisions.append(precision_score(y[val_idx], preds))

    print(f"Seuil {thresh} → Recall: {np.mean(recalls):.2f} ± {np.std(recalls):.2f}, Precision: {np.mean(precisions):.2f} ± {np.std(precisions):.2f}")

Seuil 0.1 → Recall: 1.00 ± 0.00, Precision: 0.50 ± 0.00
Seuil 0.2 → Recall: 0.89 ± 0.11, Precision: 0.66 ± 0.10
Seuil 0.3 → Recall: 0.74 ± 0.17, Precision: 0.69 ± 0.10
Seuil 0.4 → Recall: 0.74 ± 0.17, Precision: 0.72 ± 0.12
Seuil 0.5 → Recall: 0.74 ± 0.17, Precision: 0.76 ± 0.10


In [12]:
importances = pd.Series(model.feature_importances_, index=features_names).sort_values(ascending=False)
importances

n_fix_trial    1.0
dtype: float32

In [13]:
result = permutation_importance(model, X_test, y_test, n_repeats=30, random_state=42)

perm_imp = pd.Series(result.importances_mean, index=features_names).sort_values(ascending=False)
perm_imp

n_fix_trial    0.271429
dtype: float64

In [ ]:
from sklearn.model_selection import LeaveOneOut
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import numpy as np

# Utilise toutes les features (ou le sous-ensemble que tu veux tester)
features_names = dataset.drop(columns=["sid", "class_id"]).columns
X = dataset[features_names].values
y = dataset["class_id"].values

loo = LeaveOneOut()
y_true, y_pred = [], []

for train_idx, test_idx in loo.split(X):
    # Pipeline : standardisation DANS la boucle pour éviter le leakage
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", xgb.XGBClassifier(
            learning_rate=0.05,
            max_depth=2,
            n_estimators=50,
            reg_lambda=1.0,
            reg_alpha=0.1,
            min_child_weight=3,
            scale_pos_weight=(y[tr_idx] == 0).sum() / (y[tr_idx] == 1).sum(),
            random_state=42
        ))
    ])
    pipe.fit(X[train_idx], y[train_idx])
    y_pred.append(pipe.predict(X[test_idx])[0])
    y_true.append(y[test_idx][0])

print(f"LOO Accuracy  : {accuracy_score(y_true, y_pred):.4f}")
print(f"LOO Recall    : {recall_score(y_true, y_pred):.4f}")
print(f"LOO Precision : {precision_score(y_true, y_pred):.4f}")

LOO Accuracy  : 0.7286
LOO Recall    : 0.7143
LOO Precision : 0.7353
